In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn.conv import GATConv, GCNConv, SplineConv
from torch_geometric.nn.norm import GraphNorm
from torch_geometric.nn import Sequential
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
import torch.utils.data as torchdata
import tqdm
from math import floor
import matplotlib.pyplot as plt
from Model.MENDR.MENDR import MENDR_model
from Model.MENDR.Autoencoder.MENDREncoder import MENDRPatchEncoder
from Model.MENDR.mAtt.mAtt import E2R
from Model.MENDR.mAtt.optimizer import MixOptimizer
from Model.MENDR.mAtt.spd import SPDTangentSpace
import random
import os
import math
from Datasets.datasetTUAB import WaveletTUABDataset
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
print("*" * 50)
finetune_train_dataset = WaveletTUABDataset(root="/storage/ice1/shared/bmed6780/mip_group_6/ef/TUAB-128Hz/train", frac=1.0)
print("Dataset Loaded. Length of Train Dataset: ", len(finetune_train_dataset))
finetune_eval_dataset = WaveletTUABDataset(root="/storage/ice1/shared/bmed6780/mip_group_6/ef/TUAB-128Hz/eval", frac=1.0)
print("Dataset Loaded. Length of Validation Dataset: ", len(finetune_eval_dataset))

**************************************************
Loading 2717 folders


100%|██████████| 2717/2717 [04:17<00:00, 10.56it/s]


Dataset Loaded. Length of Train Dataset:  367338
Loading 276 folders


100%|██████████| 276/276 [00:25<00:00, 10.74it/s]

Dataset Loaded. Length of Validation Dataset:  36402


In [4]:
finetune_train_loader = DataLoader(finetune_train_dataset, batch_size=128, num_workers=32, shuffle=True, persistent_workers=True)
finetune_eval_loader = DataLoader(finetune_eval_dataset, batch_size=128, num_workers=32, shuffle=True, persistent_workers=True)
print(len(finetune_train_loader), len(finetune_eval_loader))
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']

2870 285


In [17]:
autoencoder = MENDRPatchEncoder(
    num_channels=19,
    sampling_rate=128,
    super_patch_seconds=2,
    delta_encoded_h=4,
    theta_encoded_h=4,
    alpha_encoded_h=8,
    beta_encoded_h=16,
    gamma_encoded_h=32,
    n_gnn_transformer_layers=4,
    num_subjects=2717,
    hidden_encoded_ratio=1,
    device=device)
for band, encoder_decoder in autoencoder.encoder_decoders.items():
	encoder_decoder.disableDecoder()
#autoencoder.load_state_dict(torch.load("./checkpoint/0cc63521e430439aba138c2875707b34_9_42/autoencoder_weights.pth", weights_only=True), strict=False)
for param in autoencoder.parameters():
    param.requires_grad = False # Freeze the autoencoder
autoencoder.eval().to(device)
print(f"Encoder Params: {sum(p.numel() for p in autoencoder.parameters() if p.requires_grad)}")

alpha Hidden Token Sequence Length: 16
beta Hidden Token Sequence Length: 32
delta Hidden Token Sequence Length: 8
gamma Hidden Token Sequence Length: 64
theta Hidden Token Sequence Length: 8
Encoder Params: 0


In [ ]:
data_iterator = iter(finetune_train_loader)
output_labels = []
data = next(data_iterator)
for key, value in data.items():
    if isinstance(value, torch.Tensor):
        data[key] = value.float().to(device)
patchified_inputs, encodings, _ = autoencoder.forward(data)
ground_truth = data['graph'].y.to(device).float()
output_labels.append(ground_truth)
output_labels = torch.cat(output_labels, dim=0).cpu().numpy().astype(int)
print(ground_truth.shape, output_labels.shape)

torch.Size([128]) (128,)


In [ ]:
for band, encoding in encodings.items():
    print(f"{band}: {encoding.shape}")

delta: torch.Size([128, 59, 4, 5])
theta: torch.Size([128, 59, 4, 5])
alpha: torch.Size([128, 59, 8, 5])
beta: torch.Size([128, 59, 16, 5])
gamma: torch.Size([128, 59, 32, 5])


In [34]:
reducer = umap.UMAP(n_components=2)

In [2]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

In [48]:
for band in tqdm.tqdm(['delta']):
    sample = encodings[band][0]
    data = sample.reshape(sample.shape[0], -1).cpu().numpy()
    #data = np.ones(data.shape)
    embeddings = pca.fit_transform(data)
    print(f"Data Shape: {data.shape} {pca.explained_variance_ratio_.sum()}")
    fig = plt.figure(figsize=(24, 12))
    ax = fig.add_subplot()
    ax.scatter(
        embeddings[:, 0],
        embeddings[:, 1]
    )
    plt.gca().set_aspect('equal', 'datalim')
    plt.title(f'PCA projection {band}', fontsize=24)
    plt.show()

100%|██████████| 1/1 [00:00<00:00, 2668.13it/s]
